# Brier Score — Avaliação de Calibração dos Modelos

O **Brier Score** mede o erro quadrático médio entre a probabilidade prevista e o desfecho real:

$$BS = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$

- Intervalo: **[0, 1]** — menor é melhor
- Modelo perfeito: BS = 0
- Modelo sem habilidade (sempre prevê a prevalência): BS = p̄(1 − p̄)

O **Brier Skill Score (BSS)** normaliza em relação ao modelo naïve:

$$BSS = 1 - \frac{BS_{modelo}}{BS_{ref}}$$

- BSS = 1 → perfeito | BSS = 0 → sem habilidade | BSS < 0 → pior que naïve

**Nenhum modelo é alterado** — análise apenas sobre os `_predicoes.parquet` existentes.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss

OUTPUT_MET = '../../output/metricas'
OUTPUT_PLT = '../../output/plots'
os.makedirs(OUTPUT_PLT, exist_ok=True)

LABEL_MAP = {
    'logistic_regression_baseline_tuned': 'LR (tuned)',
    'lightgbm_baseline_tuned':            'LightGBM (tuned)',
    'xgboost_baseline_tuned':             'XGBoost (tuned)',
    'random_forest_baseline':             'Random Forest',
    'decision_tree_baseline':             'Decision Tree',
}

## 1. Carregamento das predições

In [ ]:
predicoes = {}
for base, label in LABEL_MAP.items():
    path = os.path.join(OUTPUT_MET, f'{base}_predicoes.parquet')
    if not os.path.exists(path):
        print(f'[AVISO] arquivo não encontrado: {path}')
        continue
    df = pd.read_parquet(path)
    predicoes[base] = {
        'label':   label,
        'y_true':  df['y_true'].values,
        'y_proba': df['y_proba'].values,
    }

print(f'{len(predicoes)} modelos carregados: {[v["label"] for v in predicoes.values()]}')

## 2. Brier Score bruto

Quanto menor o valor, mais próximas as probabilidades previstas estão do desfecho real.

In [ ]:
rows = []
for base, info in predicoes.items():
    bs = brier_score_loss(info['y_true'], info['y_proba'])
    rows.append({'modelo': info['label'], 'brier_score': round(bs, 6)})

df_bs = pd.DataFrame(rows).sort_values('brier_score').reset_index(drop=True)
display(df_bs)

## 3. Brier Skill Score (BSS)

Compara o Brier Score de cada modelo ao modelo naïve que sempre prevê a prevalência global.
Um BSS positivo indica que o modelo agrega valor em relação ao baseline naïve.

In [ ]:
# Usa o y_true do primeiro modelo (todos compartilham o mesmo conjunto de teste)
y_true_ref = next(iter(predicoes.values()))['y_true']
prevalencia = y_true_ref.mean()
bs_naive    = brier_score_loss(y_true_ref, np.full_like(y_true_ref, prevalencia, dtype=float))

print(f'Prevalência de óbito: {prevalencia:.4f}')
print(f'Brier Score (naïve):  {bs_naive:.6f}')

rows_bss = []
for base, info in predicoes.items():
    bs  = brier_score_loss(info['y_true'], info['y_proba'])
    bss = 1 - bs / bs_naive
    rows_bss.append({
        'modelo':       info['label'],
        'brier_score':  round(bs, 6),
        'bss':          round(bss, 4),
    })

df_bss = pd.DataFrame(rows_bss).sort_values('bss', ascending=False).reset_index(drop=True)
display(df_bss)

## 4. Visualização

In [ ]:
df_plot = df_bss.sort_values('brier_score')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Brier Score ---
ax = axes[0]
cores_bs = ['#2ecc71' if v < 0.05 else '#e67e22' if v < 0.10 else '#e74c3c'
            for v in df_plot['brier_score']]
bars = ax.barh(df_plot['modelo'], df_plot['brier_score'], color=cores_bs, alpha=0.85)
ax.axvline(bs_naive, color='gray', linestyle='--', alpha=0.7, label=f'Naïve ({bs_naive:.4f})')
for bar, val in zip(bars, df_plot['brier_score']):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height() / 2,
            f'{val:.5f}', va='center', fontsize=9)
ax.set_xlabel('Brier Score (menor = melhor)')
ax.set_title('Brier Score por Modelo', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# --- Brier Skill Score ---
df_plot_bss = df_bss.sort_values('bss')
ax = axes[1]
cores_bss = ['#2ecc71' if v > 0.30 else '#e67e22' if v > 0.10 else '#e74c3c'
             for v in df_plot_bss['bss']]
bars = ax.barh(df_plot_bss['modelo'], df_plot_bss['bss'], color=cores_bss, alpha=0.85)
ax.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Sem habilidade (0)')
for bar, val in zip(bars, df_plot_bss['bss']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Brier Skill Score (maior = melhor)')
ax.set_title('Brier Skill Score por Modelo', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Calibração — Brier Score e Brier Skill Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'brier_score.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_plot_bss = df_bss.sort_values('bss')

fig, ax = plt.subplots(figsize=(8, 4))

cores_bss = ['#2ecc71' if v > 0.30 else '#e67e22' if v > 0.10 else '#e74c3c'
             for v in df_plot_bss['bss']]
bars = ax.barh(df_plot_bss['modelo'], df_plot_bss['bss'], color=cores_bss, alpha=0.85)

ax.axvline(0, color='gray', linestyle='--', alpha=0.7,
           label=f'Baseline (0)  |  BS naïve = {bs_naive:.4f}')

for bar, val in zip(bars, df_plot_bss['bss']):
    offset = 0.02 if val >= 0 else -0.02
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', ha=ha, fontsize=9)

ax.set_xlabel('Brier Skill Score')
ax.set_title('Brier Skill Score por Modelo', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(left=df_plot_bss['bss'].min() * 1.12, right=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'brier_skill_score.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Salvamento

In [ ]:
out_path = os.path.join(OUTPUT_MET, 'brier_score.csv')
df_bss.to_csv(out_path, index=False)
print(f'Salvo: {out_path}')